# Drone Forensic → MISP Event

This notebook walks through the pipeline **one artifact at a time**, showing for each:

1. what the extractor returns (the in-memory dict)
2. which MISP object template we use
3. how each extracted field maps to an object attribute (`object_relation → value`)

All encoder functions are imported from `misp/drone_to_misp.py` — the CLI uses the exact same functions, so running this notebook produces an equivalent MISP Event.

Switch `CASE` below to run against `forensic-01` or `forensic-02`.

In [ ]:
import inspect
import sys
from pathlib import Path

from IPython.display import Code

REPO = Path().resolve().parent
sys.path.insert(0, str(REPO / 'misp'))

import drone_to_misp
from pymisp import MISPEvent

def show_src(fn):
    return Code(inspect.getsource(fn), language='python')

def show_attrs(obj):
    fs = getattr(obj, "first_seen", None)
    ls = getattr(obj, "last_seen", None)
    bracket = f" first_seen={fs} last_seen={ls}" if fs or ls else ""
    print(f"<MISPObject name={obj.name!r} uuid={obj.uuid[:8]} attrs={len(obj.attributes)} refs={len(obj.references)}{bracket}>")
    for a in obj.attributes:
        v = a.value
        if isinstance(v, str) and len(v) > 60:
            v = v[:57] + "..."
        print(f"  {a.object_relation:<22} = {v}")
    for r in obj.references:
        print(f"  --[{r.relationship_type}]--> {r.referenced_uuid[:8]}")

In [ ]:
CASE = 'forensic-01'

case_dir = REPO / 'workshop' / CASE
fc_path   = next(case_dir.glob('fc_*.bin'), None)
bb_path   = next(case_dir.glob('blackbox_*.bin'), None)
elrs_path = next(
    (p for pattern in ('rx_*.bin', 'rx_*.hex', 'tx_*.bin', 'tx_*.hex')
       for p in case_dir.glob(pattern)),
    None,
)

print('FC      :', fc_path.name)
print('Blackbox:', bb_path.name)
print('ELRS    :', elrs_path.name)

## 1. Extract — intermediate dict

`run_extraction` wraps the three existing extractors (`fc/src/`, `blackbox/`, `elrs/src/`) and returns an in-memory dict keyed by artifact type. Missing artifacts are simply absent. **Nothing is cached to disk** — this dict is the direct input to the encoder.

In [ ]:
extracted = drone_to_misp.run_extraction(fc_path, bb_path, elrs_path)
print("Top-level keys:", list(extracted.keys()))

## 2. Create an empty Event

One case → one `MISPEvent`. We populate it by calling the encoder's per-artifact builders and inspecting each resulting object. `build_event` runs this same sequence end-to-end — we reproduce it step by step for pedagogy.

In [ ]:
event = MISPEvent()
event.info = f"Drone forensic — {CASE}"
event.distribution = 0
print(event.info)

## 3. Flight controller dump

From a single ROM dump we derive up to three MISP objects:

| Object         | Describes                                            |
|----------------|------------------------------------------------------|
| `iot-firmware` | the firmware **artifact** (raw dump as attachment, hashes, build version) |
| `uav`          | the **drone** as a whole (anchor of the graph) — also carries board info (`chipset`, `variant` = OS family) |
| `gpx`          | INAV mission waypoints, rendered in-memory (INAV only) |

### 3.1 What the FC extractor returns

YARA rules in `fc/yara/` fingerprint the firmware family, then the board/chipset. The FLASH_CONFIG region (STM32 memory map entry keyed by the chipset) is sliced out of the dump and decoded to recover the craft name, pilot name, and (on INAV) the mission waypoints.

In [ ]:
fc = extracted.get('fc', {})
print('file     :', {k: fc['file'][k] for k in ('filename','md5','size')})
print('firmware :', fc.get('firmware'))
print('craft    :', fc.get('craft_name'))
print('pilot    :', fc.get('pilot_name'))
waypoints = fc.get('waypoints', [])
if waypoints:
    print('waypoints:', len(waypoints))
    for wp in waypoints:
        print(' - ', {key: value for key, value in wp.items() if key != 'action_name'})
else:
    print('No waypoint')

### 3.2 `iot-firmware` — the firmware artifact (carrying the raw dump)

Describes the firmware blob itself: the raw bytes are base64-embedded via the `firmware` attribute (not a separate `file` object — `iot-firmware` already covers filename, hashes, size, and carries the payload). `format` is `raw` for the FC dump; `version` carries the YARA build macro when present.

In [ ]:
show_src(drone_to_misp._build_iot_firmware)

In [ ]:
iot_firmware = drone_to_misp._build_iot_firmware(
    fc["file"],
    fmt="raw",
    version=fc.get("firmware", {}).get("build_macro") or None,
)
event.add_object(iot_firmware)
show_attrs(iot_firmware)

### 3.3 `uav` — the drone (anchor)

Every other object references `uav`. When both FC and blackbox are present, **FC is authoritative** — blackbox only fills fields FC didn't produce (typically `log_start_datetime` / `firmware_version`). `_merge_craft_info` applies that dedup rule.

The `uav` template requires one of `model` / `manufacturer` / `serial-number`. We populate `model` from the FC target (falling back to the craft name), keep the user-chosen craft name in `craftname`, and record the board's `chipset` (STM32 part) and `variant` (firmware family: INAV / BETAFLIGHT / ARDUPILOT).

In [ ]:
show_src(drone_to_misp._merge_craft_info)

In [ ]:
show_src(drone_to_misp._build_uav_object)

In [ ]:
craft_info = drone_to_misp._merge_craft_info(fc, extracted.get("blackbox"))
print("merged craft_info:", craft_info)
print()
uav = drone_to_misp._build_uav_object(craft_info, fc)
event.add_object(uav)
show_attrs(uav)

### 3.4 `gpx` — waypoints as an in-memory GPX trace

INAV mission waypoints are rendered to GPX **in-memory** (not from a `.gpx` file on disk) and embedded as an attachment. If the FC has no waypoints (Betaflight, or empty INAV mission), this object is skipped and we fall back to the blackbox trace in §4.

In [ ]:
show_src(drone_to_misp._build_gpx_object)

In [ ]:
import hashlib

fc_gpx = None
if fc.get("gpx_text"):
    gpx_bytes = fc["gpx_text"].encode("utf-8")
    fc_gpx = drone_to_misp._build_gpx_object(
        fc["gpx_text"],
        filename=f"{Path(fc['file']['filename']).stem}_waypoints.gpx",
        hashes={
            "md5":    hashlib.md5(gpx_bytes).hexdigest(),
            "sha256": hashlib.sha256(gpx_bytes).hexdigest(),
            "size":   len(gpx_bytes),
        },
        waypoint_count=len(fc.get("waypoints", [])),
    )
    event.add_object(fc_gpx)
    show_attrs(fc_gpx)
else:
    print("No FC GPX for this case — will try blackbox next.")

### 3.5 Planned mission map

If the FC carried waypoints, render them on a folium map. This is the *planned* mission — the actual flown path (if any) is plotted per flight in §4.3.

In [ ]:
import folium

fc_points = [
    (wp["latitude"], wp["longitude"], f"WP{wp['index']:02d} {wp.get('action_name','')}")
    for wp in fc.get("waypoints", [])
    if wp["latitude"] != 0 or wp["longitude"] != 0
]
if fc_points:
    fc_map = folium.Map(location=[fc_points[0][0], fc_points[0][1]], zoom_start=16)
    folium.PolyLine(
        [(lat, lon) for lat, lon, _ in fc_points],
        color="blue", weight=3, opacity=0.7, tooltip="planned mission",
    ).add_to(fc_map)
    for lat, lon, tip in fc_points:
        folium.Marker(
            [lat, lon], tooltip=tip,
            icon=folium.Icon(color="blue", icon="flag"),
        ).add_to(fc_map)
    display(fc_map)
else:
    print("No FC mission waypoints — nothing to plot.")

## 4. Blackbox log

Produces a `file` (the dump), optionally a `gpx` (if FC didn't already provide one), and up to two `geolocation` objects for arming / disarming positions.

In [ ]:
bb = extracted.get("blackbox", {})
print("file :", {k: bb['file'][k] for k in ('filename','md5','size')})
for i, log in enumerate(bb.get("logs", [])[:3]):
    print(f"log #{i+1}: fw={log.get('firmware_revision')!r} craft={log.get('craft_name')!r}"
          f" start={log.get('log_start_datetime')!r} gps={log.get('has_gps')}")
    for k in ("arming_coord", "disarmed_coord"):
        if log.get(k):
            print(f"   {k}: {log[k]}")

In [ ]:
blackbox_file = drone_to_misp._build_file_object(bb["file"])
event.add_object(blackbox_file)
show_attrs(blackbox_file)

### 4.1 `gpx` per flight log — the actual flown path

The blackbox can contain **multiple flights**; the decoder writes one GPX per flight. We emit one `gpx` object per flight so each can be referenced on its own. The FC waypoints `gpx` (if any) stays alongside — it describes the *planned* mission, these describe what was *actually* flown.

In [ ]:
blackbox_gpx_by_index = {}
for idx, gpx_info in bb.get("gpx_by_index", {}).items():
    gpx_obj = drone_to_misp._build_gpx_object(
        gpx_info["gpx_text"],
        filename=gpx_info["gpx_filename"],
        hashes=gpx_info["gpx_hashes"],
        waypoint_count=0,
    )
    event.add_object(gpx_obj)
    blackbox_gpx_by_index[idx] = gpx_obj
    print(f"log #{idx}:")
    show_attrs(gpx_obj)
    print()
if not blackbox_gpx_by_index:
    print("No blackbox GPX attached (no GPS-bearing logs).")

### 4.2 `geolocation` — arming / disarming points

One object per fix, **per flight log**. The `text` attribute distinguishes `arming position` from `disarming position`; each pair will be wired (in §6) to *its own* flight's `gpx` via `starts` / `ends`.

In [ ]:
show_src(drone_to_misp._build_geolocations)

In [ ]:
geolocation_objs = drone_to_misp._build_geolocations(bb)
print(f"{len(geolocation_objs)} geolocation object(s)\n")
for idx, kind, geo in geolocation_objs:
    event.add_object(geo)
    print(f"log #{idx} — {kind}")
    show_attrs(geo)
    print()

### 4.3 Flight traces map

One folium map per flight log: the full GPS trace (decoded from the per-log GPX) plus the arming (green) and disarming (black) markers — the two `geolocation` objects for that flight.

In [ ]:
import re
import folium

logs_by_index = {l["log_index"]: l for l in bb.get("logs", [])}

if bb.get("gpx_by_index"):
    for idx, gpx_info in bb["gpx_by_index"].items():
        pts = [
            (float(lat), float(lon))
            for lat, lon in re.findall(
                r'<trkpt[^>]*lat="([^"]+)"[^>]*lon="([^"]+)"',
                gpx_info["gpx_text"],
            )
        ]
        if not pts:
            continue
        flight_map = folium.Map(location=list(pts[0]), zoom_start=16)
        folium.PolyLine(
            pts, color="red", weight=3, opacity=0.8,
            tooltip=f"flight #{idx} — {len(pts)} points",
        ).add_to(flight_map)
        log = logs_by_index.get(idx, {})
        for key, label, color, icon in (
            ("arming_coord",   "Arming",    "green", "play"),
            ("disarmed_coord", "Disarming", "black", "stop"),
        ):
            coord = log.get(key)
            if coord:
                folium.Marker(
                    [float(coord["lat"]), float(coord["lon"])],
                    tooltip=f"{label} (flight #{idx})",
                    icon=folium.Icon(color=color, icon=icon),
                ).add_to(flight_map)
        print(f"Flight #{idx}:")
        display(flight_map)
else:
    print("No blackbox flight traces to plot.")

## 5. ELRS receiver config

| Extracted                    | Object              | Attribute                                                      |
|------------------------------|---------------------|----------------------------------------------------------------|
| raw bytes, hashes, format    | `iot-firmware`      | `filename`, `md5`, `sha256`, `size-in-bytes`, `format`, `firmware` (attachment) |
| `wifi_ssid`, `wifi_password` | `wifi-connection`   | `ssid`, `password`                                             |
| `uid` (6 bytes)              | `remote-controller` | `model`=`ExpressLRS receiver`, `serial-number`, `remote-controller-id` |

The ELRS dump is a firmware image like the FC dump — same object template. When the acquired artifact is Intel HEX (`.hex`), `format` is set to `"Intel hex"` in the event itself so the provenance is visible; the extractor still feeds the decoded bytes to the JSON scanner underneath.

`wifi-connection` is only built if the config actually contained SSID/password — newer ELRS firmwares may not expose them in the dump.

In [ ]:
elrs = extracted.get("elrs", {})
for e in elrs.get("entries", []):
    uid = e.get("uid")
    if not any((uid, e.get("wifi_ssid"), e.get("wifi_password"))):
        continue
    uid_str = "-".join(f"{b:02X}" for b in uid) if uid else None
    print(f"uid={uid_str}  ssid={e.get('wifi_ssid')!r}  password={e.get('wifi_password')!r}")

In [ ]:
elrs_firmware = drone_to_misp._build_iot_firmware(
    elrs["file"], fmt=elrs.get("format", "raw"),
)
event.add_object(elrs_firmware)
show_attrs(elrs_firmware)

In [ ]:
show_src(drone_to_misp._build_elrs_objects)

In [ ]:
wifi_obj, rc_obj = drone_to_misp._build_elrs_objects(elrs)
if wifi_obj is not None:
    event.add_object(wifi_obj)
    show_attrs(wifi_obj)
    print()
else:
    print("No wifi-connection: SSID/password not present in this dump.\n")
if rc_obj is not None:
    event.add_object(rc_obj)
    show_attrs(rc_obj)

## 6. Reference graph

All objects hang off `uav`. `_add_references` applies the canonical edge set (skipping edges whose endpoints don't exist). The FC waypoints GPX represents the *planned* mission (`plans`); each blackbox flight GPX represents what was actually flown (`navigated`). Per-flight arming/disarming geolocations bind to *their own* flight's gpx.

```
uav                  --plans-->        gpx (FC waypoints)
uav                  --navigated-->    gpx (blackbox flight)   [× N logs]
uav                  --contains-->     iot-firmware (FC)
uav                  --contains-->     iot-firmware (ELRS)
uav                  --contains-->     file (blackbox)
uav                  --connects-to-->  wifi-connection
remote-controller    --controls-->     uav
geolocation (arming) --starts-->       gpx (its own flight)
geolocation (disarm) --ends-->         gpx (its own flight)
```

In [ ]:
show_src(drone_to_misp._add_references)

In [ ]:
drone_to_misp._add_references(
    uav=uav,
    iot_firmware=iot_firmware, fc_gpx=fc_gpx,
    blackbox_file=blackbox_file,
    blackbox_gpx_by_index=blackbox_gpx_by_index,
    geolocation_objs=geolocation_objs,
    elrs_firmware=elrs_firmware, wifi_obj=wifi_obj, rc_obj=rc_obj,
)

by_uuid = {o.uuid: o for o in event.objects}
for obj in event.objects:
    for r in obj.references:
        tgt = by_uuid.get(r.referenced_uuid)
        print(f"{obj.name:<18} --[{r.relationship_type}]--> {tgt.name if tgt else '?'}")

## 7. Final event

In [ ]:
print(f"{len(event.objects)} MISP objects in event {event.info!r}\n")
for obj in event.objects:
    print(f"  - {obj.name:<18} ({len(obj.attributes)} attrs, {len(obj.references)} refs)")

In [ ]:
out_path = REPO / "workshop" / f"event-{CASE}.json"
out_path.write_text(event.to_json(indent=2), encoding="utf-8")
print("Wrote", out_path)

## 8. Push our Event to MISP

In [ ]:
from pymisp import PyMISP

misp_url = '_YOUR_MISP_INSTANCE_URL_'
authentication_key = '_YOUR_MISP_AUTHENTICATION_KEY_'

misp = PyMISP(misp_url, authentication_key, ssl=True)
misp_event = misp.add_event(event, pythonify=True)
print(f'Generated MISP Event {event.id}')